# 15. Domain exceptions, retries, and resilience

**Level:** Intermediate

**Learning outcome:** Preserve error causes, bound retries, expose backoff as a callable object, and reason about breaker state.

**Prerequisites:** 03-04 and 13: invariants, dependencies, and cleanup.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](14_Network_Domain_Models_and_Parsing.ipynb) | [Next notebook](16_Strategy_Factory_Adapter_Facade_and_Decorator.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Intermediate

**Learning goal:** Report domain failures, preserve causes, and model retries without hiding errors.

**Failure is part of the interface.** A read can time out, a parser can reject a payload, and
a policy can reject an intent. Decide what each caller may catch. Custom exceptions communicate
domain meaning; exception chaining (`raise DomainError(...) from error`) preserves the original
cause so diagnostic detail is not lost.

**Do not catch everything.** Catching Exception and returning an empty dictionary can transform
a failed read into an apparently empty configuration. Catch a narrow, expected error near a
boundary and either recover deliberately or re-raise with context. A validation failure should
usually not be retried; a transient read timeout sometimes can be.

**Retries need budgets.** Limit attempts, use real timeouts in real adapters, and distinguish
transient failures from permanent errors. Backoff and jitter reduce synchronized retry load;
the teaching examples use deterministic schedules and simulated outcomes rather than sleeps.
Retrying a write is unsafe unless the operation is idempotent or you can establish whether
the first attempt took effect.

**Circuit breakers** reject work after repeated failures and later permit a controlled trial.
A two-state sketch is not a complete production breaker. A production design needs a clock,
half-open behavior, concurrency rules, metrics, and careful reset semantics.

**Cleanup and failure reporting.** Use context managers or finally to release resources.
Expected failures in this course are caught only so their exact behavior can be asserted.
Unexpected failures must stop execution, not produce a success-shaped fallback.

**Pause and explain:** Explain which errors are retryable and why blindly retrying a configuration write is dangerous.

## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 046: Custom domain exception

**Core concept:** A dedicated exception lets a caller handle one meaningful category of failure.

**Network-engineering purpose:** A missing interface can be distinguished from an unexpected programming error.

### Read the implementation in this order
1. Define InterfaceNotFound from LookupError.
2. Raise it at the domain lookup boundary.
3. Catch only that exception.
4. Assert the error includes the missing name.

**Common mistake / boundary:** Do not catch every exception around an entire fleet and mark the whole operation successful.

**Practice after the ten worked scenarios:** Attach a hostname field to the exception while keeping the human message concise.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0451 - Custom domain exception: branch scenario

**Scenario 01/10; pattern 046; Intermediate.** This is a self-contained offline exercise. A missing interface can be distinguished from an unexpected programming error.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet1/1'` |

**Logic to trace:** Define InterfaceNotFound from LookupError. Assert the error includes the missing name. Follow the class-specific steps under Pattern 046 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown interface: missing-port
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not catch every exception around an entire fleet and mark the whole operation successful.

In [1]:
# Example 0451: Custom domain exception - branch
case = {'interface': 'Ethernet1/1'}

class InterfaceNotFound(LookupError):
    pass

class Device:
    def __init__(self, interfaces):
        self.interfaces = set(interfaces)

    def require(self, name):
        if name not in self.interfaces:
            raise InterfaceNotFound(f"unknown interface: {name}")
        return name

device = Device([case["interface"]])
assert device.require(case["interface"]) == case["interface"]
try:
    device.require("missing-port")
except InterfaceNotFound as error:
    assert "missing-port" in str(error)
    print(str(error))
else:
    raise AssertionError("missing interface accepted")


unknown interface: missing-port


### Example 0452 - Custom domain exception: campus scenario

**Scenario 02/10; pattern 046; Intermediate.** This is a self-contained offline exercise. A missing interface can be distinguished from an unexpected programming error.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet2/1'` |

**Logic to trace:** Define InterfaceNotFound from LookupError. Assert the error includes the missing name. Follow the class-specific steps under Pattern 046 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown interface: missing-port
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not catch every exception around an entire fleet and mark the whole operation successful.

In [2]:
# Example 0452: Custom domain exception - campus
case = {'interface': 'Ethernet2/1'}

class InterfaceNotFound(LookupError):
    pass

class Device:
    def __init__(self, interfaces):
        self.interfaces = set(interfaces)

    def require(self, name):
        if name not in self.interfaces:
            raise InterfaceNotFound(f"unknown interface: {name}")
        return name

device = Device([case["interface"]])
assert device.require(case["interface"]) == case["interface"]
try:
    device.require("missing-port")
except InterfaceNotFound as error:
    assert "missing-port" in str(error)
    print(str(error))
else:
    raise AssertionError("missing interface accepted")


unknown interface: missing-port


### Example 0453 - Custom domain exception: dc scenario

**Scenario 03/10; pattern 046; Intermediate.** This is a self-contained offline exercise. A missing interface can be distinguished from an unexpected programming error.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet3/1'` |

**Logic to trace:** Define InterfaceNotFound from LookupError. Assert the error includes the missing name. Follow the class-specific steps under Pattern 046 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown interface: missing-port
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not catch every exception around an entire fleet and mark the whole operation successful.

In [3]:
# Example 0453: Custom domain exception - dc
case = {'interface': 'Ethernet3/1'}

class InterfaceNotFound(LookupError):
    pass

class Device:
    def __init__(self, interfaces):
        self.interfaces = set(interfaces)

    def require(self, name):
        if name not in self.interfaces:
            raise InterfaceNotFound(f"unknown interface: {name}")
        return name

device = Device([case["interface"]])
assert device.require(case["interface"]) == case["interface"]
try:
    device.require("missing-port")
except InterfaceNotFound as error:
    assert "missing-port" in str(error)
    print(str(error))
else:
    raise AssertionError("missing interface accepted")


unknown interface: missing-port


### Example 0454 - Custom domain exception: wan scenario

**Scenario 04/10; pattern 046; Intermediate.** This is a self-contained offline exercise. A missing interface can be distinguished from an unexpected programming error.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet4/1'` |

**Logic to trace:** Define InterfaceNotFound from LookupError. Assert the error includes the missing name. Follow the class-specific steps under Pattern 046 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown interface: missing-port
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not catch every exception around an entire fleet and mark the whole operation successful.

In [4]:
# Example 0454: Custom domain exception - wan
case = {'interface': 'Ethernet4/1'}

class InterfaceNotFound(LookupError):
    pass

class Device:
    def __init__(self, interfaces):
        self.interfaces = set(interfaces)

    def require(self, name):
        if name not in self.interfaces:
            raise InterfaceNotFound(f"unknown interface: {name}")
        return name

device = Device([case["interface"]])
assert device.require(case["interface"]) == case["interface"]
try:
    device.require("missing-port")
except InterfaceNotFound as error:
    assert "missing-port" in str(error)
    print(str(error))
else:
    raise AssertionError("missing interface accepted")


unknown interface: missing-port


### Example 0455 - Custom domain exception: lab scenario

**Scenario 05/10; pattern 046; Intermediate.** This is a self-contained offline exercise. A missing interface can be distinguished from an unexpected programming error.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet5/1'` |

**Logic to trace:** Define InterfaceNotFound from LookupError. Assert the error includes the missing name. Follow the class-specific steps under Pattern 046 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown interface: missing-port
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not catch every exception around an entire fleet and mark the whole operation successful.

In [5]:
# Example 0455: Custom domain exception - lab
case = {'interface': 'Ethernet5/1'}

class InterfaceNotFound(LookupError):
    pass

class Device:
    def __init__(self, interfaces):
        self.interfaces = set(interfaces)

    def require(self, name):
        if name not in self.interfaces:
            raise InterfaceNotFound(f"unknown interface: {name}")
        return name

device = Device([case["interface"]])
assert device.require(case["interface"]) == case["interface"]
try:
    device.require("missing-port")
except InterfaceNotFound as error:
    assert "missing-port" in str(error)
    print(str(error))
else:
    raise AssertionError("missing interface accepted")


unknown interface: missing-port


### Example 0456 - Custom domain exception: wireless scenario

**Scenario 06/10; pattern 046; Intermediate.** This is a self-contained offline exercise. A missing interface can be distinguished from an unexpected programming error.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet6/1'` |

**Logic to trace:** Define InterfaceNotFound from LookupError. Assert the error includes the missing name. Follow the class-specific steps under Pattern 046 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown interface: missing-port
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not catch every exception around an entire fleet and mark the whole operation successful.

In [6]:
# Example 0456: Custom domain exception - wireless
case = {'interface': 'Ethernet6/1'}

class InterfaceNotFound(LookupError):
    pass

class Device:
    def __init__(self, interfaces):
        self.interfaces = set(interfaces)

    def require(self, name):
        if name not in self.interfaces:
            raise InterfaceNotFound(f"unknown interface: {name}")
        return name

device = Device([case["interface"]])
assert device.require(case["interface"]) == case["interface"]
try:
    device.require("missing-port")
except InterfaceNotFound as error:
    assert "missing-port" in str(error)
    print(str(error))
else:
    raise AssertionError("missing interface accepted")


unknown interface: missing-port


### Example 0457 - Custom domain exception: remote scenario

**Scenario 07/10; pattern 046; Intermediate.** This is a self-contained offline exercise. A missing interface can be distinguished from an unexpected programming error.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet7/1'` |

**Logic to trace:** Define InterfaceNotFound from LookupError. Assert the error includes the missing name. Follow the class-specific steps under Pattern 046 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown interface: missing-port
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not catch every exception around an entire fleet and mark the whole operation successful.

In [7]:
# Example 0457: Custom domain exception - remote
case = {'interface': 'Ethernet7/1'}

class InterfaceNotFound(LookupError):
    pass

class Device:
    def __init__(self, interfaces):
        self.interfaces = set(interfaces)

    def require(self, name):
        if name not in self.interfaces:
            raise InterfaceNotFound(f"unknown interface: {name}")
        return name

device = Device([case["interface"]])
assert device.require(case["interface"]) == case["interface"]
try:
    device.require("missing-port")
except InterfaceNotFound as error:
    assert "missing-port" in str(error)
    print(str(error))
else:
    raise AssertionError("missing interface accepted")


unknown interface: missing-port


### Example 0458 - Custom domain exception: core scenario

**Scenario 08/10; pattern 046; Intermediate.** This is a self-contained offline exercise. A missing interface can be distinguished from an unexpected programming error.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet8/1'` |

**Logic to trace:** Define InterfaceNotFound from LookupError. Assert the error includes the missing name. Follow the class-specific steps under Pattern 046 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown interface: missing-port
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not catch every exception around an entire fleet and mark the whole operation successful.

In [8]:
# Example 0458: Custom domain exception - core
case = {'interface': 'Ethernet8/1'}

class InterfaceNotFound(LookupError):
    pass

class Device:
    def __init__(self, interfaces):
        self.interfaces = set(interfaces)

    def require(self, name):
        if name not in self.interfaces:
            raise InterfaceNotFound(f"unknown interface: {name}")
        return name

device = Device([case["interface"]])
assert device.require(case["interface"]) == case["interface"]
try:
    device.require("missing-port")
except InterfaceNotFound as error:
    assert "missing-port" in str(error)
    print(str(error))
else:
    raise AssertionError("missing interface accepted")


unknown interface: missing-port


### Example 0459 - Custom domain exception: mgmt scenario

**Scenario 09/10; pattern 046; Intermediate.** This is a self-contained offline exercise. A missing interface can be distinguished from an unexpected programming error.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet9/1'` |

**Logic to trace:** Define InterfaceNotFound from LookupError. Assert the error includes the missing name. Follow the class-specific steps under Pattern 046 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown interface: missing-port
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not catch every exception around an entire fleet and mark the whole operation successful.

In [9]:
# Example 0459: Custom domain exception - mgmt
case = {'interface': 'Ethernet9/1'}

class InterfaceNotFound(LookupError):
    pass

class Device:
    def __init__(self, interfaces):
        self.interfaces = set(interfaces)

    def require(self, name):
        if name not in self.interfaces:
            raise InterfaceNotFound(f"unknown interface: {name}")
        return name

device = Device([case["interface"]])
assert device.require(case["interface"]) == case["interface"]
try:
    device.require("missing-port")
except InterfaceNotFound as error:
    assert "missing-port" in str(error)
    print(str(error))
else:
    raise AssertionError("missing interface accepted")


unknown interface: missing-port


### Example 0460 - Custom domain exception: backup scenario

**Scenario 10/10; pattern 046; Intermediate.** This is a self-contained offline exercise. A missing interface can be distinguished from an unexpected programming error.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `interface` | `'Ethernet10/1'` |

**Logic to trace:** Define InterfaceNotFound from LookupError. Assert the error includes the missing name. Follow the class-specific steps under Pattern 046 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
unknown interface: missing-port
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Do not catch every exception around an entire fleet and mark the whole operation successful.

In [10]:
# Example 0460: Custom domain exception - backup
case = {'interface': 'Ethernet10/1'}

class InterfaceNotFound(LookupError):
    pass

class Device:
    def __init__(self, interfaces):
        self.interfaces = set(interfaces)

    def require(self, name):
        if name not in self.interfaces:
            raise InterfaceNotFound(f"unknown interface: {name}")
        return name

device = Device([case["interface"]])
assert device.require(case["interface"]) == case["interface"]
try:
    device.require("missing-port")
except InterfaceNotFound as error:
    assert "missing-port" in str(error)
    print(str(error))
else:
    raise AssertionError("missing interface accepted")


unknown interface: missing-port


## Pattern 047: Preserve a low-level cause

**Core concept:** Exception chaining adds domain context without discarding the original error.

**Network-engineering purpose:** A telemetry conversion failure should identify the field and retain the original ValueError.

### Read the implementation in this order
1. Attempt one narrow conversion.
2. Catch ValueError only.
3. Raise ParseError using from error.
4. Check both domain message and __cause__.

**Common mistake / boundary:** The final error should not dump secrets or entire sensitive payloads.

**Practice after the ten worked scenarios:** Add a supported numeric value path and a separate TypeError policy for None.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0461 - Preserve a low-level cause: branch scenario

**Scenario 01/10; pattern 047; Intermediate.** This is a self-contained offline exercise. A telemetry conversion failure should identify the field and retain the original ValueError.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `1000` |

**Logic to trace:** Attempt one narrow conversion. Check both domain message and __cause__. Follow the class-specific steps under Pattern 047 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
invalid counter field ValueError
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The final error should not dump secrets or entire sensitive payloads.

In [11]:
# Example 0461: Preserve a low-level cause - branch
case = {'counter': 1000}

class ParseError(ValueError):
    pass

class CounterParser:
    def parse(self, text):
        try:
            return int(text)
        except ValueError as error:
            raise ParseError("invalid counter field") from error

parser = CounterParser()
assert parser.parse(str(case["counter"])) == case["counter"]
try:
    parser.parse("not-a-counter")
except ParseError as error:
    assert isinstance(error.__cause__, ValueError)
    print(str(error), type(error.__cause__).__name__)
else:
    raise AssertionError("parse failure hidden")


invalid counter field ValueError


### Example 0462 - Preserve a low-level cause: campus scenario

**Scenario 02/10; pattern 047; Intermediate.** This is a self-contained offline exercise. A telemetry conversion failure should identify the field and retain the original ValueError.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `2000` |

**Logic to trace:** Attempt one narrow conversion. Check both domain message and __cause__. Follow the class-specific steps under Pattern 047 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
invalid counter field ValueError
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The final error should not dump secrets or entire sensitive payloads.

In [12]:
# Example 0462: Preserve a low-level cause - campus
case = {'counter': 2000}

class ParseError(ValueError):
    pass

class CounterParser:
    def parse(self, text):
        try:
            return int(text)
        except ValueError as error:
            raise ParseError("invalid counter field") from error

parser = CounterParser()
assert parser.parse(str(case["counter"])) == case["counter"]
try:
    parser.parse("not-a-counter")
except ParseError as error:
    assert isinstance(error.__cause__, ValueError)
    print(str(error), type(error.__cause__).__name__)
else:
    raise AssertionError("parse failure hidden")


invalid counter field ValueError


### Example 0463 - Preserve a low-level cause: dc scenario

**Scenario 03/10; pattern 047; Intermediate.** This is a self-contained offline exercise. A telemetry conversion failure should identify the field and retain the original ValueError.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `3000` |

**Logic to trace:** Attempt one narrow conversion. Check both domain message and __cause__. Follow the class-specific steps under Pattern 047 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
invalid counter field ValueError
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The final error should not dump secrets or entire sensitive payloads.

In [13]:
# Example 0463: Preserve a low-level cause - dc
case = {'counter': 3000}

class ParseError(ValueError):
    pass

class CounterParser:
    def parse(self, text):
        try:
            return int(text)
        except ValueError as error:
            raise ParseError("invalid counter field") from error

parser = CounterParser()
assert parser.parse(str(case["counter"])) == case["counter"]
try:
    parser.parse("not-a-counter")
except ParseError as error:
    assert isinstance(error.__cause__, ValueError)
    print(str(error), type(error.__cause__).__name__)
else:
    raise AssertionError("parse failure hidden")


invalid counter field ValueError


### Example 0464 - Preserve a low-level cause: wan scenario

**Scenario 04/10; pattern 047; Intermediate.** This is a self-contained offline exercise. A telemetry conversion failure should identify the field and retain the original ValueError.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `4000` |

**Logic to trace:** Attempt one narrow conversion. Check both domain message and __cause__. Follow the class-specific steps under Pattern 047 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
invalid counter field ValueError
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The final error should not dump secrets or entire sensitive payloads.

In [14]:
# Example 0464: Preserve a low-level cause - wan
case = {'counter': 4000}

class ParseError(ValueError):
    pass

class CounterParser:
    def parse(self, text):
        try:
            return int(text)
        except ValueError as error:
            raise ParseError("invalid counter field") from error

parser = CounterParser()
assert parser.parse(str(case["counter"])) == case["counter"]
try:
    parser.parse("not-a-counter")
except ParseError as error:
    assert isinstance(error.__cause__, ValueError)
    print(str(error), type(error.__cause__).__name__)
else:
    raise AssertionError("parse failure hidden")


invalid counter field ValueError


### Example 0465 - Preserve a low-level cause: lab scenario

**Scenario 05/10; pattern 047; Intermediate.** This is a self-contained offline exercise. A telemetry conversion failure should identify the field and retain the original ValueError.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `5000` |

**Logic to trace:** Attempt one narrow conversion. Check both domain message and __cause__. Follow the class-specific steps under Pattern 047 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
invalid counter field ValueError
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The final error should not dump secrets or entire sensitive payloads.

In [15]:
# Example 0465: Preserve a low-level cause - lab
case = {'counter': 5000}

class ParseError(ValueError):
    pass

class CounterParser:
    def parse(self, text):
        try:
            return int(text)
        except ValueError as error:
            raise ParseError("invalid counter field") from error

parser = CounterParser()
assert parser.parse(str(case["counter"])) == case["counter"]
try:
    parser.parse("not-a-counter")
except ParseError as error:
    assert isinstance(error.__cause__, ValueError)
    print(str(error), type(error.__cause__).__name__)
else:
    raise AssertionError("parse failure hidden")


invalid counter field ValueError


### Example 0466 - Preserve a low-level cause: wireless scenario

**Scenario 06/10; pattern 047; Intermediate.** This is a self-contained offline exercise. A telemetry conversion failure should identify the field and retain the original ValueError.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `6000` |

**Logic to trace:** Attempt one narrow conversion. Check both domain message and __cause__. Follow the class-specific steps under Pattern 047 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
invalid counter field ValueError
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The final error should not dump secrets or entire sensitive payloads.

In [16]:
# Example 0466: Preserve a low-level cause - wireless
case = {'counter': 6000}

class ParseError(ValueError):
    pass

class CounterParser:
    def parse(self, text):
        try:
            return int(text)
        except ValueError as error:
            raise ParseError("invalid counter field") from error

parser = CounterParser()
assert parser.parse(str(case["counter"])) == case["counter"]
try:
    parser.parse("not-a-counter")
except ParseError as error:
    assert isinstance(error.__cause__, ValueError)
    print(str(error), type(error.__cause__).__name__)
else:
    raise AssertionError("parse failure hidden")


invalid counter field ValueError


### Example 0467 - Preserve a low-level cause: remote scenario

**Scenario 07/10; pattern 047; Intermediate.** This is a self-contained offline exercise. A telemetry conversion failure should identify the field and retain the original ValueError.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `7000` |

**Logic to trace:** Attempt one narrow conversion. Check both domain message and __cause__. Follow the class-specific steps under Pattern 047 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
invalid counter field ValueError
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The final error should not dump secrets or entire sensitive payloads.

In [17]:
# Example 0467: Preserve a low-level cause - remote
case = {'counter': 7000}

class ParseError(ValueError):
    pass

class CounterParser:
    def parse(self, text):
        try:
            return int(text)
        except ValueError as error:
            raise ParseError("invalid counter field") from error

parser = CounterParser()
assert parser.parse(str(case["counter"])) == case["counter"]
try:
    parser.parse("not-a-counter")
except ParseError as error:
    assert isinstance(error.__cause__, ValueError)
    print(str(error), type(error.__cause__).__name__)
else:
    raise AssertionError("parse failure hidden")


invalid counter field ValueError


### Example 0468 - Preserve a low-level cause: core scenario

**Scenario 08/10; pattern 047; Intermediate.** This is a self-contained offline exercise. A telemetry conversion failure should identify the field and retain the original ValueError.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `8000` |

**Logic to trace:** Attempt one narrow conversion. Check both domain message and __cause__. Follow the class-specific steps under Pattern 047 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
invalid counter field ValueError
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The final error should not dump secrets or entire sensitive payloads.

In [18]:
# Example 0468: Preserve a low-level cause - core
case = {'counter': 8000}

class ParseError(ValueError):
    pass

class CounterParser:
    def parse(self, text):
        try:
            return int(text)
        except ValueError as error:
            raise ParseError("invalid counter field") from error

parser = CounterParser()
assert parser.parse(str(case["counter"])) == case["counter"]
try:
    parser.parse("not-a-counter")
except ParseError as error:
    assert isinstance(error.__cause__, ValueError)
    print(str(error), type(error.__cause__).__name__)
else:
    raise AssertionError("parse failure hidden")


invalid counter field ValueError


### Example 0469 - Preserve a low-level cause: mgmt scenario

**Scenario 09/10; pattern 047; Intermediate.** This is a self-contained offline exercise. A telemetry conversion failure should identify the field and retain the original ValueError.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `9000` |

**Logic to trace:** Attempt one narrow conversion. Check both domain message and __cause__. Follow the class-specific steps under Pattern 047 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
invalid counter field ValueError
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The final error should not dump secrets or entire sensitive payloads.

In [19]:
# Example 0469: Preserve a low-level cause - mgmt
case = {'counter': 9000}

class ParseError(ValueError):
    pass

class CounterParser:
    def parse(self, text):
        try:
            return int(text)
        except ValueError as error:
            raise ParseError("invalid counter field") from error

parser = CounterParser()
assert parser.parse(str(case["counter"])) == case["counter"]
try:
    parser.parse("not-a-counter")
except ParseError as error:
    assert isinstance(error.__cause__, ValueError)
    print(str(error), type(error.__cause__).__name__)
else:
    raise AssertionError("parse failure hidden")


invalid counter field ValueError


### Example 0470 - Preserve a low-level cause: backup scenario

**Scenario 10/10; pattern 047; Intermediate.** This is a self-contained offline exercise. A telemetry conversion failure should identify the field and retain the original ValueError.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `10000` |

**Logic to trace:** Attempt one narrow conversion. Check both domain message and __cause__. Follow the class-specific steps under Pattern 047 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
invalid counter field ValueError
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** The final error should not dump secrets or entire sensitive payloads.

In [20]:
# Example 0470: Preserve a low-level cause - backup
case = {'counter': 10000}

class ParseError(ValueError):
    pass

class CounterParser:
    def parse(self, text):
        try:
            return int(text)
        except ValueError as error:
            raise ParseError("invalid counter field") from error

parser = CounterParser()
assert parser.parse(str(case["counter"])) == case["counter"]
try:
    parser.parse("not-a-counter")
except ParseError as error:
    assert isinstance(error.__cause__, ValueError)
    print(str(error), type(error.__cause__).__name__)
else:
    raise AssertionError("parse failure hidden")


invalid counter field ValueError


## Pattern 048: Bounded retry of a simulated read

**Core concept:** Retry only a declared transient error and stop at a fixed budget.

**Network-engineering purpose:** A read collector can recover from temporary lab timeouts without an infinite loop.

### Read the implementation in this order
1. Inject an operation with deterministic outcomes.
2. Attempt at most the configured budget.
3. Catch only TimeoutError and re-raise the final one.
4. Check successful retry count against the simulated failures.

**Common mistake / boundary:** This is a read-only simulation; applying the same policy blindly to writes can duplicate changes.

**Practice after the ten worked scenarios:** Add an exhausted-budget case and prove the final TimeoutError propagates.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0471 - Bounded retry of a simulated read: branch scenario

**Scenario 01/10; pattern 048; Intermediate.** This is a self-contained offline exercise. A read collector can recover from temporary lab timeouts without an infinite loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `failures` | `0` |

**Logic to trace:** Inject an operation with deterministic outcomes. Check successful retry count against the simulated failures. Follow the class-specific steps under Pattern 048 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed attempts: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a read-only simulation; applying the same policy blindly to writes can duplicate changes.

In [21]:
# Example 0471: Bounded retry of a simulated read - branch
case = {'failures': 0}

class RetryReader:
    def __init__(self, attempts):
        if attempts < 1:
            raise ValueError("attempt budget must be positive")
        self.attempts = attempts

    def read(self, operation):
        for attempt in range(1, self.attempts + 1):
            try:
                return operation()
            except TimeoutError:
                if attempt == self.attempts:
                    raise

calls = []

def operation():
    calls.append("read")
    if len(calls) <= case["failures"]:
        raise TimeoutError("simulated timeout")
    return "observed"

result = RetryReader(case["failures"] + 1).read(operation)
assert result == "observed" and len(calls) == case["failures"] + 1
print(result, "attempts:", len(calls))


observed attempts: 1


### Example 0472 - Bounded retry of a simulated read: campus scenario

**Scenario 02/10; pattern 048; Intermediate.** This is a self-contained offline exercise. A read collector can recover from temporary lab timeouts without an infinite loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `failures` | `1` |

**Logic to trace:** Inject an operation with deterministic outcomes. Check successful retry count against the simulated failures. Follow the class-specific steps under Pattern 048 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed attempts: 2
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a read-only simulation; applying the same policy blindly to writes can duplicate changes.

In [22]:
# Example 0472: Bounded retry of a simulated read - campus
case = {'failures': 1}

class RetryReader:
    def __init__(self, attempts):
        if attempts < 1:
            raise ValueError("attempt budget must be positive")
        self.attempts = attempts

    def read(self, operation):
        for attempt in range(1, self.attempts + 1):
            try:
                return operation()
            except TimeoutError:
                if attempt == self.attempts:
                    raise

calls = []

def operation():
    calls.append("read")
    if len(calls) <= case["failures"]:
        raise TimeoutError("simulated timeout")
    return "observed"

result = RetryReader(case["failures"] + 1).read(operation)
assert result == "observed" and len(calls) == case["failures"] + 1
print(result, "attempts:", len(calls))


observed attempts: 2


### Example 0473 - Bounded retry of a simulated read: dc scenario

**Scenario 03/10; pattern 048; Intermediate.** This is a self-contained offline exercise. A read collector can recover from temporary lab timeouts without an infinite loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `failures` | `2` |

**Logic to trace:** Inject an operation with deterministic outcomes. Check successful retry count against the simulated failures. Follow the class-specific steps under Pattern 048 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed attempts: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a read-only simulation; applying the same policy blindly to writes can duplicate changes.

In [23]:
# Example 0473: Bounded retry of a simulated read - dc
case = {'failures': 2}

class RetryReader:
    def __init__(self, attempts):
        if attempts < 1:
            raise ValueError("attempt budget must be positive")
        self.attempts = attempts

    def read(self, operation):
        for attempt in range(1, self.attempts + 1):
            try:
                return operation()
            except TimeoutError:
                if attempt == self.attempts:
                    raise

calls = []

def operation():
    calls.append("read")
    if len(calls) <= case["failures"]:
        raise TimeoutError("simulated timeout")
    return "observed"

result = RetryReader(case["failures"] + 1).read(operation)
assert result == "observed" and len(calls) == case["failures"] + 1
print(result, "attempts:", len(calls))


observed attempts: 3


### Example 0474 - Bounded retry of a simulated read: wan scenario

**Scenario 04/10; pattern 048; Intermediate.** This is a self-contained offline exercise. A read collector can recover from temporary lab timeouts without an infinite loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `failures` | `3` |

**Logic to trace:** Inject an operation with deterministic outcomes. Check successful retry count against the simulated failures. Follow the class-specific steps under Pattern 048 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed attempts: 4
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a read-only simulation; applying the same policy blindly to writes can duplicate changes.

In [24]:
# Example 0474: Bounded retry of a simulated read - wan
case = {'failures': 3}

class RetryReader:
    def __init__(self, attempts):
        if attempts < 1:
            raise ValueError("attempt budget must be positive")
        self.attempts = attempts

    def read(self, operation):
        for attempt in range(1, self.attempts + 1):
            try:
                return operation()
            except TimeoutError:
                if attempt == self.attempts:
                    raise

calls = []

def operation():
    calls.append("read")
    if len(calls) <= case["failures"]:
        raise TimeoutError("simulated timeout")
    return "observed"

result = RetryReader(case["failures"] + 1).read(operation)
assert result == "observed" and len(calls) == case["failures"] + 1
print(result, "attempts:", len(calls))


observed attempts: 4


### Example 0475 - Bounded retry of a simulated read: lab scenario

**Scenario 05/10; pattern 048; Intermediate.** This is a self-contained offline exercise. A read collector can recover from temporary lab timeouts without an infinite loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `failures` | `0` |

**Logic to trace:** Inject an operation with deterministic outcomes. Check successful retry count against the simulated failures. Follow the class-specific steps under Pattern 048 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed attempts: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a read-only simulation; applying the same policy blindly to writes can duplicate changes.

In [25]:
# Example 0475: Bounded retry of a simulated read - lab
case = {'failures': 0}

class RetryReader:
    def __init__(self, attempts):
        if attempts < 1:
            raise ValueError("attempt budget must be positive")
        self.attempts = attempts

    def read(self, operation):
        for attempt in range(1, self.attempts + 1):
            try:
                return operation()
            except TimeoutError:
                if attempt == self.attempts:
                    raise

calls = []

def operation():
    calls.append("read")
    if len(calls) <= case["failures"]:
        raise TimeoutError("simulated timeout")
    return "observed"

result = RetryReader(case["failures"] + 1).read(operation)
assert result == "observed" and len(calls) == case["failures"] + 1
print(result, "attempts:", len(calls))


observed attempts: 1


### Example 0476 - Bounded retry of a simulated read: wireless scenario

**Scenario 06/10; pattern 048; Intermediate.** This is a self-contained offline exercise. A read collector can recover from temporary lab timeouts without an infinite loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `failures` | `1` |

**Logic to trace:** Inject an operation with deterministic outcomes. Check successful retry count against the simulated failures. Follow the class-specific steps under Pattern 048 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed attempts: 2
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a read-only simulation; applying the same policy blindly to writes can duplicate changes.

In [26]:
# Example 0476: Bounded retry of a simulated read - wireless
case = {'failures': 1}

class RetryReader:
    def __init__(self, attempts):
        if attempts < 1:
            raise ValueError("attempt budget must be positive")
        self.attempts = attempts

    def read(self, operation):
        for attempt in range(1, self.attempts + 1):
            try:
                return operation()
            except TimeoutError:
                if attempt == self.attempts:
                    raise

calls = []

def operation():
    calls.append("read")
    if len(calls) <= case["failures"]:
        raise TimeoutError("simulated timeout")
    return "observed"

result = RetryReader(case["failures"] + 1).read(operation)
assert result == "observed" and len(calls) == case["failures"] + 1
print(result, "attempts:", len(calls))


observed attempts: 2


### Example 0477 - Bounded retry of a simulated read: remote scenario

**Scenario 07/10; pattern 048; Intermediate.** This is a self-contained offline exercise. A read collector can recover from temporary lab timeouts without an infinite loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `failures` | `2` |

**Logic to trace:** Inject an operation with deterministic outcomes. Check successful retry count against the simulated failures. Follow the class-specific steps under Pattern 048 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed attempts: 3
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a read-only simulation; applying the same policy blindly to writes can duplicate changes.

In [27]:
# Example 0477: Bounded retry of a simulated read - remote
case = {'failures': 2}

class RetryReader:
    def __init__(self, attempts):
        if attempts < 1:
            raise ValueError("attempt budget must be positive")
        self.attempts = attempts

    def read(self, operation):
        for attempt in range(1, self.attempts + 1):
            try:
                return operation()
            except TimeoutError:
                if attempt == self.attempts:
                    raise

calls = []

def operation():
    calls.append("read")
    if len(calls) <= case["failures"]:
        raise TimeoutError("simulated timeout")
    return "observed"

result = RetryReader(case["failures"] + 1).read(operation)
assert result == "observed" and len(calls) == case["failures"] + 1
print(result, "attempts:", len(calls))


observed attempts: 3


### Example 0478 - Bounded retry of a simulated read: core scenario

**Scenario 08/10; pattern 048; Intermediate.** This is a self-contained offline exercise. A read collector can recover from temporary lab timeouts without an infinite loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `failures` | `3` |

**Logic to trace:** Inject an operation with deterministic outcomes. Check successful retry count against the simulated failures. Follow the class-specific steps under Pattern 048 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed attempts: 4
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a read-only simulation; applying the same policy blindly to writes can duplicate changes.

In [28]:
# Example 0478: Bounded retry of a simulated read - core
case = {'failures': 3}

class RetryReader:
    def __init__(self, attempts):
        if attempts < 1:
            raise ValueError("attempt budget must be positive")
        self.attempts = attempts

    def read(self, operation):
        for attempt in range(1, self.attempts + 1):
            try:
                return operation()
            except TimeoutError:
                if attempt == self.attempts:
                    raise

calls = []

def operation():
    calls.append("read")
    if len(calls) <= case["failures"]:
        raise TimeoutError("simulated timeout")
    return "observed"

result = RetryReader(case["failures"] + 1).read(operation)
assert result == "observed" and len(calls) == case["failures"] + 1
print(result, "attempts:", len(calls))


observed attempts: 4


### Example 0479 - Bounded retry of a simulated read: mgmt scenario

**Scenario 09/10; pattern 048; Intermediate.** This is a self-contained offline exercise. A read collector can recover from temporary lab timeouts without an infinite loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `failures` | `0` |

**Logic to trace:** Inject an operation with deterministic outcomes. Check successful retry count against the simulated failures. Follow the class-specific steps under Pattern 048 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed attempts: 1
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a read-only simulation; applying the same policy blindly to writes can duplicate changes.

In [29]:
# Example 0479: Bounded retry of a simulated read - mgmt
case = {'failures': 0}

class RetryReader:
    def __init__(self, attempts):
        if attempts < 1:
            raise ValueError("attempt budget must be positive")
        self.attempts = attempts

    def read(self, operation):
        for attempt in range(1, self.attempts + 1):
            try:
                return operation()
            except TimeoutError:
                if attempt == self.attempts:
                    raise

calls = []

def operation():
    calls.append("read")
    if len(calls) <= case["failures"]:
        raise TimeoutError("simulated timeout")
    return "observed"

result = RetryReader(case["failures"] + 1).read(operation)
assert result == "observed" and len(calls) == case["failures"] + 1
print(result, "attempts:", len(calls))


observed attempts: 1


### Example 0480 - Bounded retry of a simulated read: backup scenario

**Scenario 10/10; pattern 048; Intermediate.** This is a self-contained offline exercise. A read collector can recover from temporary lab timeouts without an infinite loop.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `failures` | `1` |

**Logic to trace:** Inject an operation with deterministic outcomes. Check successful retry count against the simulated failures. Follow the class-specific steps under Pattern 048 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed attempts: 2
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This is a read-only simulation; applying the same policy blindly to writes can duplicate changes.

In [30]:
# Example 0480: Bounded retry of a simulated read - backup
case = {'failures': 1}

class RetryReader:
    def __init__(self, attempts):
        if attempts < 1:
            raise ValueError("attempt budget must be positive")
        self.attempts = attempts

    def read(self, operation):
        for attempt in range(1, self.attempts + 1):
            try:
                return operation()
            except TimeoutError:
                if attempt == self.attempts:
                    raise

calls = []

def operation():
    calls.append("read")
    if len(calls) <= case["failures"]:
        raise TimeoutError("simulated timeout")
    return "observed"

result = RetryReader(case["failures"] + 1).read(operation)
assert result == "observed" and len(calls) == case["failures"] + 1
print(result, "attempts:", len(calls))


observed attempts: 2


## Pattern 049: Deterministic exponential backoff schedule

**Core concept:** A policy object computes a capped delay rather than sleeping in a unit example.

**Network-engineering purpose:** Retry timing can be tested separately from transport behavior.

### Read the implementation in this order
1. Validate a nonnegative attempt index.
2. Compute base times two to the index.
3. Use __call__ to expose the same computation through policy(index).
4. Check monotonicity and the maximum across a small schedule.

**Common mistake / boundary:** A production schedule usually needs jitter and a total deadline, not only exponential delays.

**Practice after the ten worked scenarios:** Inject a deterministic random source and add bounded jitter without changing the transport.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0481 - Deterministic exponential backoff schedule: branch scenario

**Scenario 01/10; pattern 049; Intermediate.** This is a self-contained offline exercise. Retry timing can be tested separately from transport behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `base_delay` | `1` |

**Logic to trace:** Validate a nonnegative attempt index. Check monotonicity and the maximum across a small schedule. Follow the class-specific steps under Pattern 049 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[1, 2, 4, 8, 8, 8]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A production schedule usually needs jitter and a total deadline, not only exponential delays.

In [31]:
# Example 0481: Deterministic exponential backoff schedule - branch
case = {'base_delay': 1}

class Backoff:
    def __init__(self, base, cap):
        if base <= 0 or cap < base:
            raise ValueError("invalid backoff bounds")
        self.base, self.cap = base, cap

    def delay(self, attempt):
        if type(attempt) is not int or attempt < 0:
            raise ValueError("attempt index must be nonnegative")
        return min(self.cap, self.base * 2 ** attempt)

    def __call__(self, attempt):
        return self.delay(attempt)

policy = Backoff(case["base_delay"], 8)
delays = [policy(index) for index in range(6)]
assert delays == sorted(delays) and max(delays) <= 8
assert delays[0] == case["base_delay"]
assert policy(0) == policy.delay(0)
print(delays)


[1, 2, 4, 8, 8, 8]


### Example 0482 - Deterministic exponential backoff schedule: campus scenario

**Scenario 02/10; pattern 049; Intermediate.** This is a self-contained offline exercise. Retry timing can be tested separately from transport behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `base_delay` | `2` |

**Logic to trace:** Validate a nonnegative attempt index. Check monotonicity and the maximum across a small schedule. Follow the class-specific steps under Pattern 049 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[2, 4, 8, 8, 8, 8]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A production schedule usually needs jitter and a total deadline, not only exponential delays.

In [32]:
# Example 0482: Deterministic exponential backoff schedule - campus
case = {'base_delay': 2}

class Backoff:
    def __init__(self, base, cap):
        if base <= 0 or cap < base:
            raise ValueError("invalid backoff bounds")
        self.base, self.cap = base, cap

    def delay(self, attempt):
        if type(attempt) is not int or attempt < 0:
            raise ValueError("attempt index must be nonnegative")
        return min(self.cap, self.base * 2 ** attempt)

    def __call__(self, attempt):
        return self.delay(attempt)

policy = Backoff(case["base_delay"], 8)
delays = [policy(index) for index in range(6)]
assert delays == sorted(delays) and max(delays) <= 8
assert delays[0] == case["base_delay"]
assert policy(0) == policy.delay(0)
print(delays)


[2, 4, 8, 8, 8, 8]


### Example 0483 - Deterministic exponential backoff schedule: dc scenario

**Scenario 03/10; pattern 049; Intermediate.** This is a self-contained offline exercise. Retry timing can be tested separately from transport behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `base_delay` | `0.5` |

**Logic to trace:** Validate a nonnegative attempt index. Check monotonicity and the maximum across a small schedule. Follow the class-specific steps under Pattern 049 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[0.5, 1.0, 2.0, 4.0, 8, 8]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A production schedule usually needs jitter and a total deadline, not only exponential delays.

In [33]:
# Example 0483: Deterministic exponential backoff schedule - dc
case = {'base_delay': 0.5}

class Backoff:
    def __init__(self, base, cap):
        if base <= 0 or cap < base:
            raise ValueError("invalid backoff bounds")
        self.base, self.cap = base, cap

    def delay(self, attempt):
        if type(attempt) is not int or attempt < 0:
            raise ValueError("attempt index must be nonnegative")
        return min(self.cap, self.base * 2 ** attempt)

    def __call__(self, attempt):
        return self.delay(attempt)

policy = Backoff(case["base_delay"], 8)
delays = [policy(index) for index in range(6)]
assert delays == sorted(delays) and max(delays) <= 8
assert delays[0] == case["base_delay"]
assert policy(0) == policy.delay(0)
print(delays)


[0.5, 1.0, 2.0, 4.0, 8, 8]


### Example 0484 - Deterministic exponential backoff schedule: wan scenario

**Scenario 04/10; pattern 049; Intermediate.** This is a self-contained offline exercise. Retry timing can be tested separately from transport behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `base_delay` | `1` |

**Logic to trace:** Validate a nonnegative attempt index. Check monotonicity and the maximum across a small schedule. Follow the class-specific steps under Pattern 049 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[1, 2, 4, 8, 8, 8]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A production schedule usually needs jitter and a total deadline, not only exponential delays.

In [34]:
# Example 0484: Deterministic exponential backoff schedule - wan
case = {'base_delay': 1}

class Backoff:
    def __init__(self, base, cap):
        if base <= 0 or cap < base:
            raise ValueError("invalid backoff bounds")
        self.base, self.cap = base, cap

    def delay(self, attempt):
        if type(attempt) is not int or attempt < 0:
            raise ValueError("attempt index must be nonnegative")
        return min(self.cap, self.base * 2 ** attempt)

    def __call__(self, attempt):
        return self.delay(attempt)

policy = Backoff(case["base_delay"], 8)
delays = [policy(index) for index in range(6)]
assert delays == sorted(delays) and max(delays) <= 8
assert delays[0] == case["base_delay"]
assert policy(0) == policy.delay(0)
print(delays)


[1, 2, 4, 8, 8, 8]


### Example 0485 - Deterministic exponential backoff schedule: lab scenario

**Scenario 05/10; pattern 049; Intermediate.** This is a self-contained offline exercise. Retry timing can be tested separately from transport behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `base_delay` | `2` |

**Logic to trace:** Validate a nonnegative attempt index. Check monotonicity and the maximum across a small schedule. Follow the class-specific steps under Pattern 049 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[2, 4, 8, 8, 8, 8]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A production schedule usually needs jitter and a total deadline, not only exponential delays.

In [35]:
# Example 0485: Deterministic exponential backoff schedule - lab
case = {'base_delay': 2}

class Backoff:
    def __init__(self, base, cap):
        if base <= 0 or cap < base:
            raise ValueError("invalid backoff bounds")
        self.base, self.cap = base, cap

    def delay(self, attempt):
        if type(attempt) is not int or attempt < 0:
            raise ValueError("attempt index must be nonnegative")
        return min(self.cap, self.base * 2 ** attempt)

    def __call__(self, attempt):
        return self.delay(attempt)

policy = Backoff(case["base_delay"], 8)
delays = [policy(index) for index in range(6)]
assert delays == sorted(delays) and max(delays) <= 8
assert delays[0] == case["base_delay"]
assert policy(0) == policy.delay(0)
print(delays)


[2, 4, 8, 8, 8, 8]


### Example 0486 - Deterministic exponential backoff schedule: wireless scenario

**Scenario 06/10; pattern 049; Intermediate.** This is a self-contained offline exercise. Retry timing can be tested separately from transport behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `base_delay` | `0.25` |

**Logic to trace:** Validate a nonnegative attempt index. Check monotonicity and the maximum across a small schedule. Follow the class-specific steps under Pattern 049 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[0.25, 0.5, 1.0, 2.0, 4.0, 8]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A production schedule usually needs jitter and a total deadline, not only exponential delays.

In [36]:
# Example 0486: Deterministic exponential backoff schedule - wireless
case = {'base_delay': 0.25}

class Backoff:
    def __init__(self, base, cap):
        if base <= 0 or cap < base:
            raise ValueError("invalid backoff bounds")
        self.base, self.cap = base, cap

    def delay(self, attempt):
        if type(attempt) is not int or attempt < 0:
            raise ValueError("attempt index must be nonnegative")
        return min(self.cap, self.base * 2 ** attempt)

    def __call__(self, attempt):
        return self.delay(attempt)

policy = Backoff(case["base_delay"], 8)
delays = [policy(index) for index in range(6)]
assert delays == sorted(delays) and max(delays) <= 8
assert delays[0] == case["base_delay"]
assert policy(0) == policy.delay(0)
print(delays)


[0.25, 0.5, 1.0, 2.0, 4.0, 8]

### Example 0487 - Deterministic exponential backoff schedule: remote scenario

**Scenario 07/10; pattern 049; Intermediate.** This is a self-contained offline exercise. Retry timing can be tested separately from transport behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `base_delay` | `1` |

**Logic to trace:** Validate a nonnegative attempt index. Check monotonicity and the maximum across a small schedule. Follow the class-specific steps under Pattern 049 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[1, 2, 4, 8, 8, 8]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A production schedule usually needs jitter and a total deadline, not only exponential delays.

In [37]:
# Example 0487: Deterministic exponential backoff schedule - remote
case = {'base_delay': 1}

class Backoff:
    def __init__(self, base, cap):
        if base <= 0 or cap < base:
            raise ValueError("invalid backoff bounds")
        self.base, self.cap = base, cap

    def delay(self, attempt):
        if type(attempt) is not int or attempt < 0:
            raise ValueError("attempt index must be nonnegative")
        return min(self.cap, self.base * 2 ** attempt)

    def __call__(self, attempt):
        return self.delay(attempt)

policy = Backoff(case["base_delay"], 8)
delays = [policy(index) for index in range(6)]
assert delays == sorted(delays) and max(delays) <= 8
assert delays[0] == case["base_delay"]
assert policy(0) == policy.delay(0)
print(delays)


[1, 2, 4, 8, 8, 8]


### Example 0488 - Deterministic exponential backoff schedule: core scenario

**Scenario 08/10; pattern 049; Intermediate.** This is a self-contained offline exercise. Retry timing can be tested separately from transport behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `base_delay` | `0.5` |

**Logic to trace:** Validate a nonnegative attempt index. Check monotonicity and the maximum across a small schedule. Follow the class-specific steps under Pattern 049 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[0.5, 1.0, 2.0, 4.0, 8, 8]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A production schedule usually needs jitter and a total deadline, not only exponential delays.

In [38]:
# Example 0488: Deterministic exponential backoff schedule - core
case = {'base_delay': 0.5}

class Backoff:
    def __init__(self, base, cap):
        if base <= 0 or cap < base:
            raise ValueError("invalid backoff bounds")
        self.base, self.cap = base, cap

    def delay(self, attempt):
        if type(attempt) is not int or attempt < 0:
            raise ValueError("attempt index must be nonnegative")
        return min(self.cap, self.base * 2 ** attempt)

    def __call__(self, attempt):
        return self.delay(attempt)

policy = Backoff(case["base_delay"], 8)
delays = [policy(index) for index in range(6)]
assert delays == sorted(delays) and max(delays) <= 8
assert delays[0] == case["base_delay"]
assert policy(0) == policy.delay(0)
print(delays)


[0.5, 1.0, 2.0, 4.0, 8, 8]


### Example 0489 - Deterministic exponential backoff schedule: mgmt scenario

**Scenario 09/10; pattern 049; Intermediate.** This is a self-contained offline exercise. Retry timing can be tested separately from transport behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `base_delay` | `2` |

**Logic to trace:** Validate a nonnegative attempt index. Check monotonicity and the maximum across a small schedule. Follow the class-specific steps under Pattern 049 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[2, 4, 8, 8, 8, 8]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A production schedule usually needs jitter and a total deadline, not only exponential delays.

In [39]:
# Example 0489: Deterministic exponential backoff schedule - mgmt
case = {'base_delay': 2}

class Backoff:
    def __init__(self, base, cap):
        if base <= 0 or cap < base:
            raise ValueError("invalid backoff bounds")
        self.base, self.cap = base, cap

    def delay(self, attempt):
        if type(attempt) is not int or attempt < 0:
            raise ValueError("attempt index must be nonnegative")
        return min(self.cap, self.base * 2 ** attempt)

    def __call__(self, attempt):
        return self.delay(attempt)

policy = Backoff(case["base_delay"], 8)
delays = [policy(index) for index in range(6)]
assert delays == sorted(delays) and max(delays) <= 8
assert delays[0] == case["base_delay"]
assert policy(0) == policy.delay(0)
print(delays)


[2, 4, 8, 8, 8, 8]


### Example 0490 - Deterministic exponential backoff schedule: backup scenario

**Scenario 10/10; pattern 049; Intermediate.** This is a self-contained offline exercise. Retry timing can be tested separately from transport behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `base_delay` | `1` |

**Logic to trace:** Validate a nonnegative attempt index. Check monotonicity and the maximum across a small schedule. Follow the class-specific steps under Pattern 049 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[1, 2, 4, 8, 8, 8]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** A production schedule usually needs jitter and a total deadline, not only exponential delays.

In [40]:
# Example 0490: Deterministic exponential backoff schedule - backup
case = {'base_delay': 1}

class Backoff:
    def __init__(self, base, cap):
        if base <= 0 or cap < base:
            raise ValueError("invalid backoff bounds")
        self.base, self.cap = base, cap

    def delay(self, attempt):
        if type(attempt) is not int or attempt < 0:
            raise ValueError("attempt index must be nonnegative")
        return min(self.cap, self.base * 2 ** attempt)

    def __call__(self, attempt):
        return self.delay(attempt)

policy = Backoff(case["base_delay"], 8)
delays = [policy(index) for index in range(6)]
assert delays == sorted(delays) and max(delays) <= 8
assert delays[0] == case["base_delay"]
assert policy(0) == policy.delay(0)
print(delays)


[1, 2, 4, 8, 8, 8]


## Pattern 050: Explicit two-state breaker sketch

**Core concept:** A stateful guard can stop repeated work after a failure threshold.

**Network-engineering purpose:** An offline breaker demonstrates why persistent state changes caller behavior.

### Read the implementation in this order
1. Track consecutive failures.
2. Open the breaker at the threshold.
3. Reject a subsequent call before executing its operation.
4. Reset explicitly and verify the closed state.

**Common mistake / boundary:** This teaching sketch lacks timed recovery, half-open trials, and concurrency protection.

**Practice after the ten worked scenarios:** Add an injected clock and exactly one half-open trial without suppressing its exception.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0491 - Explicit two-state breaker sketch: branch scenario

**Scenario 01/10; pattern 050; Intermediate.** This is a self-contained offline exercise. An offline breaker demonstrates why persistent state changes caller behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `attempts` | `1` |

**Logic to trace:** Track consecutive failures. Reset explicitly and verify the closed state. Follow the class-specific steps under Pattern 050 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
read circuit open
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This teaching sketch lacks timed recovery, half-open trials, and concurrency protection.

In [41]:
# Example 0491: Explicit two-state breaker sketch - branch
case = {'attempts': 1}

class CircuitOpen(RuntimeError):
    pass

class Breaker:
    def __init__(self, limit):
        if limit < 1:
            raise ValueError("limit must be positive")
        self.limit, self.failures = limit, 0

    def record_failure(self):
        self.failures += 1

    def guard(self):
        if self.failures >= self.limit:
            raise CircuitOpen("read circuit open")

    def reset(self):
        self.failures = 0

breaker = Breaker(case["attempts"])
for _ in range(case["attempts"]):
    breaker.record_failure()
try:
    breaker.guard()
except CircuitOpen as error:
    print(str(error))
else:
    raise AssertionError("open circuit not enforced")
breaker.reset()
breaker.guard()
assert breaker.failures == 0


read circuit open


### Example 0492 - Explicit two-state breaker sketch: campus scenario

**Scenario 02/10; pattern 050; Intermediate.** This is a self-contained offline exercise. An offline breaker demonstrates why persistent state changes caller behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `attempts` | `2` |

**Logic to trace:** Track consecutive failures. Reset explicitly and verify the closed state. Follow the class-specific steps under Pattern 050 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
read circuit open
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This teaching sketch lacks timed recovery, half-open trials, and concurrency protection.

In [42]:
# Example 0492: Explicit two-state breaker sketch - campus
case = {'attempts': 2}

class CircuitOpen(RuntimeError):
    pass

class Breaker:
    def __init__(self, limit):
        if limit < 1:
            raise ValueError("limit must be positive")
        self.limit, self.failures = limit, 0

    def record_failure(self):
        self.failures += 1

    def guard(self):
        if self.failures >= self.limit:
            raise CircuitOpen("read circuit open")

    def reset(self):
        self.failures = 0

breaker = Breaker(case["attempts"])
for _ in range(case["attempts"]):
    breaker.record_failure()
try:
    breaker.guard()
except CircuitOpen as error:
    print(str(error))
else:
    raise AssertionError("open circuit not enforced")
breaker.reset()
breaker.guard()
assert breaker.failures == 0


read circuit open


### Example 0493 - Explicit two-state breaker sketch: dc scenario

**Scenario 03/10; pattern 050; Intermediate.** This is a self-contained offline exercise. An offline breaker demonstrates why persistent state changes caller behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `attempts` | `3` |

**Logic to trace:** Track consecutive failures. Reset explicitly and verify the closed state. Follow the class-specific steps under Pattern 050 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
read circuit open
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This teaching sketch lacks timed recovery, half-open trials, and concurrency protection.

In [43]:
# Example 0493: Explicit two-state breaker sketch - dc
case = {'attempts': 3}

class CircuitOpen(RuntimeError):
    pass

class Breaker:
    def __init__(self, limit):
        if limit < 1:
            raise ValueError("limit must be positive")
        self.limit, self.failures = limit, 0

    def record_failure(self):
        self.failures += 1

    def guard(self):
        if self.failures >= self.limit:
            raise CircuitOpen("read circuit open")

    def reset(self):
        self.failures = 0

breaker = Breaker(case["attempts"])
for _ in range(case["attempts"]):
    breaker.record_failure()
try:
    breaker.guard()
except CircuitOpen as error:
    print(str(error))
else:
    raise AssertionError("open circuit not enforced")
breaker.reset()
breaker.guard()
assert breaker.failures == 0


read circuit open


### Example 0494 - Explicit two-state breaker sketch: wan scenario

**Scenario 04/10; pattern 050; Intermediate.** This is a self-contained offline exercise. An offline breaker demonstrates why persistent state changes caller behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `attempts` | `4` |

**Logic to trace:** Track consecutive failures. Reset explicitly and verify the closed state. Follow the class-specific steps under Pattern 050 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
read circuit open
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This teaching sketch lacks timed recovery, half-open trials, and concurrency protection.

In [44]:
# Example 0494: Explicit two-state breaker sketch - wan
case = {'attempts': 4}

class CircuitOpen(RuntimeError):
    pass

class Breaker:
    def __init__(self, limit):
        if limit < 1:
            raise ValueError("limit must be positive")
        self.limit, self.failures = limit, 0

    def record_failure(self):
        self.failures += 1

    def guard(self):
        if self.failures >= self.limit:
            raise CircuitOpen("read circuit open")

    def reset(self):
        self.failures = 0

breaker = Breaker(case["attempts"])
for _ in range(case["attempts"]):
    breaker.record_failure()
try:
    breaker.guard()
except CircuitOpen as error:
    print(str(error))
else:
    raise AssertionError("open circuit not enforced")
breaker.reset()
breaker.guard()
assert breaker.failures == 0


read circuit open


### Example 0495 - Explicit two-state breaker sketch: lab scenario

**Scenario 05/10; pattern 050; Intermediate.** This is a self-contained offline exercise. An offline breaker demonstrates why persistent state changes caller behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `attempts` | `1` |

**Logic to trace:** Track consecutive failures. Reset explicitly and verify the closed state. Follow the class-specific steps under Pattern 050 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
read circuit open
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This teaching sketch lacks timed recovery, half-open trials, and concurrency protection.

In [45]:
# Example 0495: Explicit two-state breaker sketch - lab
case = {'attempts': 1}

class CircuitOpen(RuntimeError):
    pass

class Breaker:
    def __init__(self, limit):
        if limit < 1:
            raise ValueError("limit must be positive")
        self.limit, self.failures = limit, 0

    def record_failure(self):
        self.failures += 1

    def guard(self):
        if self.failures >= self.limit:
            raise CircuitOpen("read circuit open")

    def reset(self):
        self.failures = 0

breaker = Breaker(case["attempts"])
for _ in range(case["attempts"]):
    breaker.record_failure()
try:
    breaker.guard()
except CircuitOpen as error:
    print(str(error))
else:
    raise AssertionError("open circuit not enforced")
breaker.reset()
breaker.guard()
assert breaker.failures == 0


read circuit open


### Example 0496 - Explicit two-state breaker sketch: wireless scenario

**Scenario 06/10; pattern 050; Intermediate.** This is a self-contained offline exercise. An offline breaker demonstrates why persistent state changes caller behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `attempts` | `2` |

**Logic to trace:** Track consecutive failures. Reset explicitly and verify the closed state. Follow the class-specific steps under Pattern 050 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
read circuit open
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This teaching sketch lacks timed recovery, half-open trials, and concurrency protection.

In [46]:
# Example 0496: Explicit two-state breaker sketch - wireless
case = {'attempts': 2}

class CircuitOpen(RuntimeError):
    pass

class Breaker:
    def __init__(self, limit):
        if limit < 1:
            raise ValueError("limit must be positive")
        self.limit, self.failures = limit, 0

    def record_failure(self):
        self.failures += 1

    def guard(self):
        if self.failures >= self.limit:
            raise CircuitOpen("read circuit open")

    def reset(self):
        self.failures = 0

breaker = Breaker(case["attempts"])
for _ in range(case["attempts"]):
    breaker.record_failure()
try:
    breaker.guard()
except CircuitOpen as error:
    print(str(error))
else:
    raise AssertionError("open circuit not enforced")
breaker.reset()
breaker.guard()
assert breaker.failures == 0


read circuit open


### Example 0497 - Explicit two-state breaker sketch: remote scenario

**Scenario 07/10; pattern 050; Intermediate.** This is a self-contained offline exercise. An offline breaker demonstrates why persistent state changes caller behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `attempts` | `3` |

**Logic to trace:** Track consecutive failures. Reset explicitly and verify the closed state. Follow the class-specific steps under Pattern 050 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
read circuit open
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This teaching sketch lacks timed recovery, half-open trials, and concurrency protection.

In [47]:
# Example 0497: Explicit two-state breaker sketch - remote
case = {'attempts': 3}

class CircuitOpen(RuntimeError):
    pass

class Breaker:
    def __init__(self, limit):
        if limit < 1:
            raise ValueError("limit must be positive")
        self.limit, self.failures = limit, 0

    def record_failure(self):
        self.failures += 1

    def guard(self):
        if self.failures >= self.limit:
            raise CircuitOpen("read circuit open")

    def reset(self):
        self.failures = 0

breaker = Breaker(case["attempts"])
for _ in range(case["attempts"]):
    breaker.record_failure()
try:
    breaker.guard()
except CircuitOpen as error:
    print(str(error))
else:
    raise AssertionError("open circuit not enforced")
breaker.reset()
breaker.guard()
assert breaker.failures == 0


read circuit open


### Example 0498 - Explicit two-state breaker sketch: core scenario

**Scenario 08/10; pattern 050; Intermediate.** This is a self-contained offline exercise. An offline breaker demonstrates why persistent state changes caller behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `attempts` | `4` |

**Logic to trace:** Track consecutive failures. Reset explicitly and verify the closed state. Follow the class-specific steps under Pattern 050 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
read circuit open
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This teaching sketch lacks timed recovery, half-open trials, and concurrency protection.

In [48]:
# Example 0498: Explicit two-state breaker sketch - core
case = {'attempts': 4}

class CircuitOpen(RuntimeError):
    pass

class Breaker:
    def __init__(self, limit):
        if limit < 1:
            raise ValueError("limit must be positive")
        self.limit, self.failures = limit, 0

    def record_failure(self):
        self.failures += 1

    def guard(self):
        if self.failures >= self.limit:
            raise CircuitOpen("read circuit open")

    def reset(self):
        self.failures = 0

breaker = Breaker(case["attempts"])
for _ in range(case["attempts"]):
    breaker.record_failure()
try:
    breaker.guard()
except CircuitOpen as error:
    print(str(error))
else:
    raise AssertionError("open circuit not enforced")
breaker.reset()
breaker.guard()
assert breaker.failures == 0


read circuit open


### Example 0499 - Explicit two-state breaker sketch: mgmt scenario

**Scenario 09/10; pattern 050; Intermediate.** This is a self-contained offline exercise. An offline breaker demonstrates why persistent state changes caller behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `attempts` | `1` |

**Logic to trace:** Track consecutive failures. Reset explicitly and verify the closed state. Follow the class-specific steps under Pattern 050 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
read circuit open
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This teaching sketch lacks timed recovery, half-open trials, and concurrency protection.

In [49]:
# Example 0499: Explicit two-state breaker sketch - mgmt
case = {'attempts': 1}

class CircuitOpen(RuntimeError):
    pass

class Breaker:
    def __init__(self, limit):
        if limit < 1:
            raise ValueError("limit must be positive")
        self.limit, self.failures = limit, 0

    def record_failure(self):
        self.failures += 1

    def guard(self):
        if self.failures >= self.limit:
            raise CircuitOpen("read circuit open")

    def reset(self):
        self.failures = 0

breaker = Breaker(case["attempts"])
for _ in range(case["attempts"]):
    breaker.record_failure()
try:
    breaker.guard()
except CircuitOpen as error:
    print(str(error))
else:
    raise AssertionError("open circuit not enforced")
breaker.reset()
breaker.guard()
assert breaker.failures == 0


read circuit open


### Example 0500 - Explicit two-state breaker sketch: backup scenario

**Scenario 10/10; pattern 050; Intermediate.** This is a self-contained offline exercise. An offline breaker demonstrates why persistent state changes caller behavior.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `attempts` | `2` |

**Logic to trace:** Track consecutive failures. Reset explicitly and verify the closed state. Follow the class-specific steps under Pattern 050 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
read circuit open
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This teaching sketch lacks timed recovery, half-open trials, and concurrency protection.

In [50]:
# Example 0500: Explicit two-state breaker sketch - backup
case = {'attempts': 2}

class CircuitOpen(RuntimeError):
    pass

class Breaker:
    def __init__(self, limit):
        if limit < 1:
            raise ValueError("limit must be positive")
        self.limit, self.failures = limit, 0

    def record_failure(self):
        self.failures += 1

    def guard(self):
        if self.failures >= self.limit:
            raise CircuitOpen("read circuit open")

    def reset(self):
        self.failures = 0

breaker = Breaker(case["attempts"])
for _ in range(case["attempts"]):
    breaker.record_failure()
try:
    breaker.guard()
except CircuitOpen as error:
    print(str(error))
else:
    raise AssertionError("open circuit not enforced")
breaker.reset()
breaker.guard()
assert breaker.failures == 0


read circuit open


## Check your understanding: Domain exceptions, retries, and resilience

**You should now be able to:** Preserve error causes, bound retries, expose backoff as a callable object, and reason about breaker state.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.